# Hierarchical MARL Portfolio Optimization with Semantic Diversification
## Week 5 Progress — Agent Implementation Begins

**Author**: Radheshyam Subedi | **Student ID**: U2829927 | **Date**: 24 Feb 2026

### Project Timeline
| Week | Phase | Status |
|------|-------|--------|
| 1–2 | Data Pipeline & Text Processing | ✅ Complete |
| 3–4 | Gymnasium Environment Development | ✅ Complete |
| **5–6** | **Agent Implementation (EIIE/PyTorch)** | **🔄 In Progress** |
| 7–8 | Training & Walk-Forward Validation | ⬜ Upcoming |
| 9–10 | Experiments & Crisis Evaluation | ⬜ Upcoming |
| 11–12 | Dissertation Writing & Submission | ⬜ Upcoming |

### Overview
This notebook implements the core components of the Hierarchical Multi-Agent Reinforcement Learning system:
1. **Data Pipeline** — 45 S&P 500 stocks with 10-K filing text for semantic analysis
2. **Gymnasium Environments** — WorkerEnv (stock selection) + ManagerEnv (sector allocation)
3. **EIIE Agent** — True Conv2d implementation from Jiang et al. (2017) *(just starting)*

### Reward Function
$$R_{total} = R_{portfolio} - \lambda \sum_{i,j} w_i w_j S_{ij} - \gamma \sum_i |w_{i,t} - w_{i,t-1}|$$

Where $S_{ij}$ is TF-IDF cosine similarity from SEC 10-K filings.

In [ ]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings, json, time
warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
print("All libraries loaded successfully.")
print(f"PyTorch version: {torch.__version__}")
print(f"Gymnasium version: {gym.__version__}")

---
## Section 1: Data Pipeline & Text Processing ✅ *(Weeks 1–2: Complete)*

### Data Sources
| Source | Purpose | Method |
|--------|---------|--------|
| **SEC EDGAR** | 10-K Item 1 (Business Description) | `sec-edgar-downloader` API |
| **Yahoo Finance** | Fallback business summaries + daily prices | `yfinance` |
| **Wikipedia** | S&P 500 constituent list + GICS sectors | Web scraping |

### Pipeline Steps
1. Scrape S&P 500 tickers from Wikipedia
2. Randomly sample 50 stocks (seed=42), download daily prices (2015–2024)
3. Extract 10-K Item 1 text from SEC EDGAR (fallback: Yahoo Finance summaries)
4. Build 45×45 TF-IDF cosine similarity matrix
5. Map GICS sector assignments for hierarchical structure

> **Note**: The data pipeline script (`load_sp500_50.py`) has already been run.
> We load pre-computed results here.

In [ ]:
# Load pre-computed Phase 3 data (generated by load_sp500_50.py)
price_df = pd.read_csv("data/raw/sp500_50_prices.csv", index_col=0, parse_dates=True)
lexical_df = pd.read_csv("data/processed/lexical_matrix_50.csv", index_col=0)
with open("data/processed/sector_map_50.json") as f:
    sector_map = json.load(f)

tickers = list(price_df.columns)
n_stocks = len(tickers)

print("=" * 60)
print("DATA LOADED SUCCESSFULLY")
print("=" * 60)
print(f"Stocks:       {n_stocks}")
print(f"Date range:   {price_df.index[0][:10]} to {price_df.index[-1][:10]}")
print(f"Trading days: {len(price_df)}")
print(f"Lexical matrix: {lexical_df.shape}")
print(f"\nStocks: {', '.join(tickers[:10])}{'...' if len(tickers) > 10 else ''}")

In [ ]:
# Build sector groups from GICS classifications
sectors = {}
for t, s in sector_map.items():
    if t in tickers:
        sectors.setdefault(s, []).append(t)

# Filter to sectors with 2+ stocks (Workers need at least 2)
sectors = {s: ts for s, ts in sectors.items() if len(ts) >= 2}
valid_tickers = [t for ts in sectors.values() for t in ts]
price_df = price_df[valid_tickers]

print(f"After filtering: {len(valid_tickers)} stocks in {len(sectors)} GICS sectors:\n")
for s, ts in sorted(sectors.items()):
    print(f"  {s} ({len(ts)} stocks): {', '.join(ts)}")

### 10-K Lexical Similarity Matrix

The lexical matrix captures **structural business similarity** from SEC 10-K filings.
Companies in the same industry (e.g., two utilities) have high similarity scores,
while unrelated businesses (e.g., a utility and a tech firm) have near-zero scores.

> **10-K filings** provide 2,000–10,000 words per company (vs ~200 from Yahoo),
> giving a much richer similarity signal for the semantic penalty.

In [ ]:
# Lexical matrix statistics
upper = lexical_df.values[np.triu_indices_from(lexical_df.values, k=1)]
print("Lexical Similarity Statistics:")
print(f"  Mean: {np.mean(upper):.4f}")
print(f"  Std:  {np.std(upper):.4f}")
print(f"  Min:  {np.min(upper):.4f}")
print(f"  Max:  {np.max(upper):.4f}")

# Check text sources
try:
    with open("data/raw/sp500_50_10k_texts.json") as f:
        text_data = json.load(f)
    src_counts = Counter(text_data["sources"].values())
    print(f"\nText Sources:")
    for src, cnt in src_counts.most_common():
        print(f"  {src}: {cnt} stocks")
except FileNotFoundError:
    print("\n(10-K text source file not found — statistics skipped)")

In [ ]:
# Heatmap visualization
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(lexical_df, cmap='YlOrRd', ax=ax, xticklabels=True, yticklabels=True,
            vmin=0, vmax=1, cbar_kws={'label': 'Cosine Similarity'})
ax.set_title("TF-IDF Cosine Similarity Matrix (45 S&P 500 Stocks — 10-K Filings)", fontsize=13)
plt.xticks(fontsize=6, rotation=90)
plt.yticks(fontsize=6)
plt.tight_layout()
plt.show()

In [ ]:
# Price data overview
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Normalized price curves for sample stocks
sample_stocks = valid_tickers[:8]
norm_prices = price_df[sample_stocks] / price_df[sample_stocks].iloc[0]
for col in norm_prices.columns:
    axes[0].plot(norm_prices.index, norm_prices[col], label=col, linewidth=1)
axes[0].set_title("Normalized Price Curves (Sample of 8 Stocks)")
axes[0].set_ylabel("Normalized Price (Start = 1.0)")
axes[0].legend(fontsize=7, ncol=2)
axes[0].grid(alpha=0.3)

# Sector distribution
sector_counts = {s: len(ts) for s, ts in sorted(sectors.items())}
colors = plt.cm.Set3(np.linspace(0, 1, len(sector_counts)))
axes[1].barh(list(sector_counts.keys()), list(sector_counts.values()), color=colors)
axes[1].set_xlabel("Number of Stocks")
axes[1].set_title("GICS Sector Distribution")
for i, (s, c) in enumerate(sector_counts.items()):
    axes[1].text(c + 0.1, i, str(c), va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## Section 2: Gymnasium Environment Development ✅ *(Weeks 3–4: Complete)*

### Manager-Worker Hierarchical Architecture

```
                    ┌──────────────────────┐
                    │    MANAGER AGENT     │
                    │  (Sector Allocator)  │
                    └──────────┬───────────┘
                               │
         Sector weights: [w_util, w_IT, w_fin, w_health, ...]
                               │
    ┌──────────┬──────────┬────┴────┬──────────┬──────────┐
    ▼          ▼          ▼         ▼          ▼          ▼
┌────────┐ ┌────────┐ ┌────────┐ ┌────────┐ ┌────────┐  ...
│ Worker │ │ Worker │ │ Worker │ │ Worker │ │ Worker │
│ Utils  │ │  IT    │ │Finance │ │Health  │ │Indust. │
└────────┘ └────────┘ └────────┘ └────────┘ └────────┘
```

### Why This Separation?

| Level | Agent | Manages | Risk Type |
|-------|-------|---------|-----------||
| **Top** | Manager | Sector allocation | **Systemic risk** (macro shocks) |
| **Bottom** | Workers | Stock selection within sector | **Idiosyncratic risk** (company-specific) |

### Final Portfolio Weights
$$w_{final,i} = w_{sector,k} \times w_{stock,i|k}$$

### WorkerEnv — Stock Selection Within a Sector

Each Worker manages stocks within a single GICS sector.

**Observation**: `[normalized_price_window (N×W), portfolio_weights (N)]`
**Action**: Continuous logits → softmax → portfolio weights
**Reward**: `log_return - λ·semantic_penalty - γ·turnover`

The **semantic penalty** from the 10-K lexical matrix penalizes holding similar stocks:
$$penalty = \sum_{i,j} w_i \cdot w_j \cdot S_{ij}$$

In [ ]:
class WorkerEnv(gym.Env):
    """Worker agent: picks stocks WITHIN a single GICS sector.
    
    The semantic penalty uses the 10-K lexical similarity matrix to
    discourage holding stocks with similar business descriptions.
    """
    metadata = {"render_modes": []}
    
    def __init__(self, price_df, lexical_matrix, tickers, window_size=30,
                 lambda_penalty=0.1, gamma_penalty=0.01):
        super().__init__()
        self.tickers = [t for t in tickers if t in price_df.columns and t in lexical_matrix.index]
        self.n_assets = len(self.tickers)
        if self.n_assets == 0:
            raise ValueError("No valid tickers for WorkerEnv")
        
        self.prices = price_df[self.tickers].values
        self.lexical_matrix = np.nan_to_num(
            lexical_matrix.loc[self.tickers, self.tickers].values, nan=0.0)
        self.window_size = window_size
        self.lambda_penalty = lambda_penalty
        self.gamma_penalty = gamma_penalty
        
        obs_dim = window_size * self.n_assets + self.n_assets
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(self.n_assets,), dtype=np.float32)
        
        self.current_step = 0
        self.portfolio_weights = np.ones(self.n_assets) / self.n_assets
        self.max_steps = len(self.prices) - window_size - 1
        self.portfolio_value = 1.0
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = self.window_size
        self.portfolio_weights = np.ones(self.n_assets) / self.n_assets
        self.portfolio_value = 1.0
        return self._get_obs(), {}
    
    def step(self, action):
        # Softmax normalization of action
        exp_a = np.exp(action - np.max(action))
        w = exp_a / np.sum(exp_a)
        
        # Portfolio return
        curr = self.prices[self.current_step]
        nxt = self.prices[self.current_step + 1]
        pr = nxt / (curr + 1e-8)
        port_ret = max(np.dot(w, pr), 1e-8)
        log_ret = np.log(port_ret)
        
        # Penalties
        sem_pen = np.dot(w.T, np.dot(self.lexical_matrix, w))
        turnover = np.sum(np.abs(w - self.portfolio_weights))
        
        # Combined reward
        reward = float(log_ret - self.lambda_penalty * sem_pen - self.gamma_penalty * turnover)
        
        self.portfolio_weights = w
        self.portfolio_value *= port_ret
        self.current_step += 1
        terminated = self.current_step >= self.max_steps
        
        return self._get_obs(), reward, terminated, False, {
            'portfolio_return': port_ret, 'semantic_penalty': sem_pen,
            'turnover': turnover, 'weights': w.copy()}
    
    def _get_obs(self):
        s = self.current_step - self.window_size
        win = self.prices[s:self.current_step]
        norm = win / (win[0, :] + 1e-8)
        return np.concatenate([norm.flatten(), self.portfolio_weights]).astype(np.float32)

print("WorkerEnv defined.")

### ManagerEnv — Sector Capital Allocation

The Manager sees **sector-level features** and outputs sector allocation weights.

**Observation**: `[sector_price_window, sector_weights, sector_similarity]`
**Action**: Continuous logits → softmax → sector weights
**Reward**: `log_return - 0.1·sim_penalty - 0.01·turnover`

In [ ]:
class ManagerEnv(gym.Env):
    """Manager agent: allocates capital across GICS sectors.
    
    Uses sector-averaged prices and intra-sector lexical similarity
    to make allocation decisions.
    """
    metadata = {"render_modes": []}
    
    def __init__(self, price_df, lexical_matrix, sectors, window_size=30):
        super().__init__()
        self.sectors = sectors
        self.sector_names = list(sectors.keys())
        self.n_sectors = len(self.sector_names)
        
        # Compute sector-level average prices
        self.sector_prices = {}
        for name, tickers in sectors.items():
            valid_t = [t for t in tickers if t in price_df.columns]
            self.sector_prices[name] = price_df[valid_t].mean(axis=1).values
        
        self.window_size = window_size
        self.all_prices = np.column_stack([self.sector_prices[s] for s in self.sector_names])
        
        # Intra-sector similarity (avg pairwise similarity within each sector)
        self.sector_sim = np.zeros(self.n_sectors)
        for i, name in enumerate(self.sector_names):
            tickers = [t for t in sectors[name] if t in lexical_matrix.index]
            if len(tickers) > 1:
                sim = lexical_matrix.loc[tickers, tickers].values
                mask = np.triu(np.ones_like(sim, dtype=bool), k=1)
                self.sector_sim[i] = np.nanmean(sim[mask])
        
        obs_dim = window_size * self.n_sectors + self.n_sectors + self.n_sectors
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(self.n_sectors,), dtype=np.float32)
        
        self.current_step = 0
        self.sector_weights = np.ones(self.n_sectors) / self.n_sectors
        self.max_steps = len(self.all_prices) - window_size - 1
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = self.window_size
        self.sector_weights = np.ones(self.n_sectors) / self.n_sectors
        return self._get_obs(), {}
    
    def step(self, action):
        exp_a = np.exp(action - np.max(action))
        w = exp_a / np.sum(exp_a)
        curr = self.all_prices[self.current_step]
        nxt = self.all_prices[self.current_step + 1]
        pr = nxt / (curr + 1e-8)
        port_ret = max(np.dot(w, pr), 1e-8)
        log_ret = np.log(port_ret)
        sim_penalty = np.dot(w, self.sector_sim)
        turnover = np.sum(np.abs(w - self.sector_weights))
        reward = float(log_ret - 0.1 * sim_penalty - 0.01 * turnover)
        self.sector_weights = w
        self.current_step += 1
        terminated = self.current_step >= self.max_steps
        return self._get_obs(), reward, terminated, False, {
            'sector_weights': w.copy(), 'sim_penalty': sim_penalty}
    
    def _get_obs(self):
        s = self.current_step - self.window_size
        win = self.all_prices[s:self.current_step]
        norm = win / (win[0, :] + 1e-8)
        return np.concatenate([norm.flatten(), self.sector_weights, self.sector_sim]).astype(np.float32)

print("ManagerEnv defined.")

### Environment Validation

We verify both environments work correctly by:
1. Creating instances with our real data
2. Checking observation/action space shapes
3. Running a few random steps to confirm the reward signal works

In [ ]:
# Test WorkerEnv with a single sector
test_sector = list(sectors.keys())[0]
test_tickers = sectors[test_sector]
print(f"Testing WorkerEnv with sector: {test_sector} ({len(test_tickers)} stocks)")

worker_env = WorkerEnv(price_df, lexical_df, test_tickers, window_size=30, lambda_penalty=0.1)
obs, info = worker_env.reset()
print(f"  Observation shape: {obs.shape}")
print(f"  Action space:      {worker_env.action_space.shape}")
print(f"  Max steps:         {worker_env.max_steps}")

# Take 5 random steps
print(f"\n  Random walk (5 steps):")
for step in range(5):
    action = worker_env.action_space.sample()
    obs, reward, terminated, truncated, info = worker_env.step(action)
    print(f"    Step {step+1}: reward={reward:.4f}, sem_penalty={info['semantic_penalty']:.4f}, "
          f"turnover={info['turnover']:.4f}")

print(f"\n  ✓ WorkerEnv is functional!")

In [ ]:
# Test ManagerEnv
print(f"Testing ManagerEnv with {len(sectors)} sectors")

manager_env = ManagerEnv(price_df, lexical_df, sectors, window_size=30)
obs, info = manager_env.reset()
print(f"  Observation shape: {obs.shape}")
print(f"  Action space:      {manager_env.action_space.shape}")
print(f"  Sectors:           {manager_env.sector_names}")
print(f"  Sector similarity: {np.round(manager_env.sector_sim, 4)}")

# Take 5 random steps
print(f"\n  Random walk (5 steps):")
for step in range(5):
    action = manager_env.action_space.sample()
    obs, reward, terminated, truncated, info = manager_env.step(action)
    sector_w = info['sector_weights']
    top_sector = manager_env.sector_names[np.argmax(sector_w)]
    print(f"    Step {step+1}: reward={reward:.4f}, top_sector={top_sector} ({np.max(sector_w):.2f})")

print(f"\n  ✓ ManagerEnv is functional!")

---
## Section 3: Agent Implementation — True EIIE Network 🔄 *(Week 5: In Progress)*

### EIIE Architecture (Jiang et al., 2017)

The **Ensemble of Identical Independent Evaluators** requires:
1. **Identical** — All assets share the same network weights
2. **Independent** — No cross-asset feature contamination during evaluation
3. **Scalable** — Architecture works for N=5 or N=500 without changes

### Conv2d Implementation

We use `Conv2d` with a **(1, 3) kernel** to satisfy these properties:
- Kernel height = 1 → **never crosses asset boundaries** (independence)
- Conv2d weight sharing → **same weights for all assets** (identical)
- Single GPU kernel call → **fully vectorized** (scalable)
- **1×1 voting layer** replaces `nn.Linear` → position-independent scoring

### Why NOT the For-Loop Approach?
| For-Loop Approach | Conv2d Approach |
|---|---|
| `for i in range(n_assets):` Conv1d loop | Single Conv2d call for all assets |
| `nn.Linear(n_assets * 32 + n_assets, 64)` — position-dependent | `nn.Conv2d(33, 1, (1,1))` — position-independent |
| O(N) sequential Python calls | Single GPU kernel, O(1) calls |
| Breaks EIIE independence (cross-asset weights) | True EIIE topology |

### Tensor Shape Flow
```
Input:  (Batch, 1, Assets, Time_Window)     e.g., (1, 1, 45, 30)
         ↓ Conv2d(1, 16, (1,3))
        (Batch, 16, Assets, Time_Window)
         ↓ Conv2d(16, 32, (1,3))
        (Batch, 32, Assets, Time_Window)
         ↓ mean(dim=3, keepdim=True)         Temporal pooling
        (Batch, 32, Assets, 1)
         ↓ cat([features, PVM], dim=1)       Append portfolio memory
        (Batch, 33, Assets, 1)
         ↓ Conv2d(33, 1, (1,1))              Voting layer
        (Batch, 1, Assets, 1)
         ↓ squeeze(3).squeeze(1)             Remove singleton dims
        (Batch, Assets)
         ↓ softmax(dim=-1)
Output: (Batch, Assets)                      Portfolio weights, sum=1
```

In [ ]:
class EIIENetwork(nn.Module):
    """True EIIE: Conv2d with (1,3) kernel evaluates all assets in parallel
    without cross-contaminating features. 1x1 voting layer produces per-asset
    scores. Scales to arbitrary N without architecture changes.

    Input shapes:
        x:   (Batch, 1, Assets, Time_Window)
        pvm: (Batch, Assets)
    Output: (Batch, Assets) -- softmax portfolio weights
    """
    def __init__(self, n_assets, window_size, num_features=1):
        super().__init__()
        self.n_assets = n_assets
        # Shared feature extraction -- kernel (1,3) never crosses asset boundaries
        self.conv1 = nn.Conv2d(num_features, 16, kernel_size=(1, 3), padding=(0, 1))
        self.conv2 = nn.Conv2d(16, 32, kernel_size=(1, 3), padding=(0, 1))
        # Voting layer -- 1x1 conv produces one score per asset (32 channels + 1 PVM = 33)
        self.vote = nn.Conv2d(33, 1, kernel_size=(1, 1))

    def forward(self, x, pvm):
        # x: (B, 1, N, W)  pvm: (B, N)
        x = F.relu(self.conv1(x))           # (B, 16, N, W)
        x = F.relu(self.conv2(x))           # (B, 32, N, W)
        x = x.mean(dim=3, keepdim=True)     # (B, 32, N, 1) -- temporal pooling
        # Append PVM as extra channel
        pvm_expanded = pvm.unsqueeze(1).unsqueeze(3)  # (B, 1, N, 1)
        x = torch.cat([x, pvm_expanded], dim=1)       # (B, 33, N, 1)
        scores = self.vote(x)                          # (B, 1, N, 1)
        scores = scores.squeeze(3).squeeze(1)          # (B, N) -- explicit dims, batch-safe
        return F.softmax(scores, dim=-1)

print("EIIENetwork defined (Conv2d -- True EIIE).")
print(f"  Parameters: {sum(p.numel() for p in EIIENetwork(45, 30).parameters()):,}")

### Verifying the EIIE Architecture

Before training, we verify three critical properties:
1. **Correct output shape** — `(Batch, Assets)` with weights summing to 1
2. **Batch safety** — Works for both single steps (B=1) and batched updates (B=64)
3. **Scalability** — Same architecture handles 5, 45, or 500 assets

In [ ]:
# Test 1: Forward pass with real dimensions
print("Test 1: Forward pass (45 stocks, window=30)")
net = EIIENetwork(n_assets=45, window_size=30)
x = torch.randn(1, 1, 45, 30)   # (Batch=1, Features=1, Assets=45, Window=30)
pvm = torch.randn(1, 45)         # (Batch=1, Assets=45)
out = net(x, pvm)
print(f"  Input:  x={list(x.shape)}, pvm={list(pvm.shape)}")
print(f"  Output: {list(out.shape)}, sum={out.sum().item():.4f}")
assert out.shape == (1, 45), f"Expected (1, 45), got {out.shape}"
assert abs(out.sum().item() - 1.0) < 1e-5
print("  ✓ PASS\n")

# Test 2: Batch dimension safety
print("Test 2: Batch safety (B=4)")
x_batch = torch.randn(4, 1, 45, 30)
pvm_batch = torch.randn(4, 45)
out_batch = net(x_batch, pvm_batch)
print(f"  Input:  x={list(x_batch.shape)}, pvm={list(pvm_batch.shape)}")
print(f"  Output: {list(out_batch.shape)}")
assert out_batch.shape == (4, 45), f"Expected (4, 45), got {out_batch.shape}"
print("  ✓ PASS\n")

# Test 3: Scalability
for n in [5, 45, 100, 500]:
    net_test = EIIENetwork(n_assets=n, window_size=30)
    x_test = torch.randn(1, 1, n, 30)
    pvm_test = torch.randn(1, n)
    out_test = net_test(x_test, pvm_test)
    params = sum(p.numel() for p in net_test.parameters())
    assert out_test.shape == (1, n)
    print(f"Test 3: N={n:>3d} → output {list(out_test.shape)}, params={params:,}  ✓")

print("\n✅ All EIIE architecture tests passed!")

### REINFORCE Training Loop

Simple policy gradient training using the REINFORCE algorithm.
The training loop reshapes the flat observation from the Gymnasium environment
into the 4D tensor expected by the Conv2d EIIE network.

In [ ]:
def train_reinforce(env, n_episodes=50, lr=1e-3, max_steps=200, verbose=False):
    """Train an EIIE agent using REINFORCE. Returns net and history."""
    n = env.action_space.shape[0]
    ws = env.window_size
    net = EIIENetwork(n, ws)
    opt = optim.Adam(net.parameters(), lr=lr)
    hist = {'rewards': [], 'penalties': [], 'values': []}
    
    for ep in range(n_episodes):
        obs, _ = env.reset()
        lps, rews, sps = [], [], []
        done, steps = False, 0
        
        while not done and steps < max_steps:
            obs_dim = ws * n
            # Reshape to (1, 1, N, W) for Conv2d EIIE
            pw = torch.FloatTensor(obs[:obs_dim].reshape(1, n, ws)).unsqueeze(0)
            pvm = torch.FloatTensor(obs[obs_dim:obs_dim+n]).unsqueeze(0)
            weights = net(pw, pvm)
            dist = torch.distributions.Normal(weights.squeeze(), 0.1)
            action = dist.sample()
            lps.append(dist.log_prob(action).sum())
            obs, reward, terminated, truncated, info = env.step(action.detach().numpy())
            done = terminated or truncated
            rews.append(reward)
            sps.append(info.get('semantic_penalty', info.get('sim_penalty', 0)))
            steps += 1
        
        # Compute discounted returns
        G, rets = 0, []
        for r in reversed(rews):
            G = r + 0.99 * G
            rets.insert(0, G)
        rets = torch.FloatTensor(rets)
        if len(rets) > 1:
            rets = (rets - rets.mean()) / (rets.std() + 1e-8)
        
        # Policy gradient update
        loss = sum(-lp * G for lp, G in zip(lps, rets))
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        
        hist['rewards'].append(sum(rews))
        hist['penalties'].append(float(np.mean(sps)))
        hist['values'].append(getattr(env, 'portfolio_value', 1.0))
        
        if verbose and (ep+1) % 10 == 0:
            print(f"  Ep {ep+1:3d} | R: {sum(rews):.4f} | Pen: {np.mean(sps):.4f}")
    
    return net, hist

print("Training function defined.")

### Quick Training Demo (Smoke Test)

We run a short training session (10 episodes) on a small subset to verify that:
1. The training loop runs without errors
2. The reward signal is sensible (not NaN, trends as expected)
3. The Conv2d EIIE integrates correctly with the Gymnasium environment

In [ ]:
# Quick demo: train on one sector for 10 episodes
demo_sector = list(sectors.keys())[0]
demo_tickers = sectors[demo_sector]
print(f"Demo: Training on {demo_sector} ({len(demo_tickers)} stocks) for 10 episodes...\n")

demo_env = WorkerEnv(price_df, lexical_df, demo_tickers, window_size=30, lambda_penalty=0.1)
demo_net, demo_hist = train_reinforce(demo_env, n_episodes=10, max_steps=50, verbose=True)

print(f"\nTraining complete!")
print(f"  Episodes:     {len(demo_hist['rewards'])}")
print(f"  Final reward:  {demo_hist['rewards'][-1]:.4f}")
print(f"  Reward trend:  {demo_hist['rewards'][0]:.4f} → {demo_hist['rewards'][-1]:.4f}")
print(f"  Avg penalty:   {np.mean(demo_hist['penalties']):.4f}")

# Plot training curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(demo_hist['rewards'], 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title(f'Demo Training Curve — {demo_sector} ({len(demo_tickers)} stocks, 10 episodes)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✓ EIIE agent integrates with WorkerEnv successfully!")

---
## Next Steps (Weeks 6–10)

### Week 6: Complete Agent Implementation
- [ ] Train full Manager agent (50 episodes, all sectors)
- [ ] Train Worker agents for each GICS sector (50 episodes each)
- [ ] Implement evaluation function with comprehensive risk metrics

### Weeks 7–8: Training & Walk-Forward Validation
- [ ] Walk-Forward validation across 4 market regimes (2019, 2020-COVID, 2021, 2022-Bear)
- [ ] Lambda ablation study (λ = 0.0, 0.1, 0.5, 1.0)
- [ ] Analyze semantic penalty impact on concentration metrics

### Weeks 9–10: Experiments & Crisis Evaluation
- [ ] Drawdown decomposition (depth, duration, recovery)
- [ ] Full risk metrics: CVaR, Sortino, Calmar, HHI, Effective N
- [ ] Crisis regime analysis (COVID crash, 2022 bear market)
- [ ] Compare with equal-weight benchmark

### Weeks 11–12: Dissertation Writing
- [ ] Results compilation and visualization
- [ ] Final report writing and submission